In [ ]:
# 🛠️ 環境セットアップ

# 共通ライブラリのインポート
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

# Google Colab環境かどうかを判定
try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

# ライブラリのセットアップ
if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    
    # 必要なパッケージをインストール
    !pip install japanize-matplotlib
    
    # GitHubからライブラリをクローン
    !git clone https://github.com/ggszk/simple-audio-programming.git
    
    # パスを追加
    sys.path.append('/content/simple-audio-programming')

    # ライブラリ設定
    import japanize_matplotlib
    
else:
    print("🔧 ローカル環境を設定中...")

    # パスを追加
    sys.path.append('..')

    # 日本語フォント設定（Mac）
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("\n🎵 Simple Audio Programming へようこそ！")
print("✅ セットアップ完了")

# 音のプログラミング 第1回: 基本概念とサイン波

**学習目標:**
- 音とプログラミングの関係を理解する
- サイン波の基本概念を学ぶ
- デジタル音声のしくみ（サンプリング）を理解する
- 周波数と音程の関係を体験する
- 最初の音を生成・再生する

**所要時間:** 90分

## 🛠️ 音声処理ヘルパー関数の定義

音声再生と波形表示のためのヘルパー関数を定義します。

In [ ]:
# 音声再生用ヘルパー関数
def play_sound(signal, title="Audio"):
    """
    Colab/Jupyter環境で音声を再生するヘルパー関数
    
    Args:
        signal: AudioSignal オブジェクト
        title: 表示用タイトル
    """
    print(f"🔊 {title} (サンプルレート: {signal.sample_rate} Hz)")
    # クリッピング防止のため振幅を0.8倍に抑える
    data = signal.data * 0.8
    return Audio(data, rate=signal.sample_rate, normalize=False)

def plot_waveform(signal, duration=0.01, title="波形"):
    """
    波形を可視化するヘルパー関数
    
    Args:
        signal: AudioSignal オブジェクト
        duration: 表示する時間長（秒）
        title: グラフのタイトル
    """
    time_samples = int(duration * signal.sample_rate)
    time_samples = min(time_samples, signal.num_samples)
    time_array = np.linspace(0, duration, time_samples)
    
    plt.figure(figsize=(12, 6))
    plt.plot(time_array, signal.data[:time_samples], 'b-', linewidth=2)
    plt.title(title, fontsize=16)
    plt.xlabel('時間 (秒)', fontsize=12)
    plt.ylabel('振幅', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.show()

print("🛠️ ヘルパー関数を読み込みました")

### 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, save_audio, AudioSignal
from audio_lib.synthesis import note_to_frequency, note_name_to_number, sawtooth_wave

## 🎵 音とは何か？

### 音の正体
- 音は **空気の振動**
- 振動が波として伝わる
- **周波数**（1秒間の振動回数）で音程が決まる
- **振幅**（振動の大きさ）で音量が決まる

### サイン波
最も基本的な音の波形は **サイン波（正弦波）**

数式: `y = sin(2π × 周波数 × 時間)`

## 🎯 実習1: 最初のサイン波を作ろう

In [ ]:
# サイン波を生成する関数の準備完了
# sine_wave(周波数, 長さ) で AudioSignal を返します

print("🎵 サイン波生成の準備完了！")

In [ ]:
# 440Hz（ラ音）のサイン波を1秒間生成
frequency = 440.0  # Hz（ヘルツ）
duration = 1.0     # 秒

# サイン波を生成
signal = sine_wave(frequency, duration)

print(f"✅ {frequency}Hzの音を{duration}秒分生成しました")
print(f"データの長さ: {len(signal)} サンプル")
print(f"最大振幅: {np.max(signal.data):.3f}")
print(f"最小振幅: {np.min(signal.data):.3f}")

In [ ]:
# 波形を可視化してみよう
# ヘルパー関数を使って最初の0.01秒分を表示
plot_waveform(signal, duration=0.01, 
              title=f'{frequency}Hz サイン波の波形')

print("📊 これがサイン波の形です！きれいな波になっていますね。")

In [ ]:
# 🔊 音を再生してみよう！
audio_player = play_sound(signal, f"{frequency}Hz サイン波")
display(audio_player)

print("🎵 これが440Hz（ラ音）の純粋なサイン波です！")
print("楽器の音とは違って、とてもシンプルな音ですね。")

## 🔬 デジタル音声とサンプリング

### 音はどうやってコンピューターに記録される？

上のセルで `len(signal)` を確認すると **44100** サンプルでした。
これは「1秒間に44100回、波の高さを測定した」という意味です。

| 用語 | 意味 | 例 |
|------|------|----|
| **サンプリング周波数** | 1秒間の測定回数 | 44100 Hz（CD品質） |
| **サンプル** | 1回の測定値 | -1.0 〜 1.0 の数値 |

### ナイキスト定理：なぜ 44100 Hz なのか？

**元の音を正確に復元するには、最高周波数の2倍以上でサンプリングする必要がある**（ナイキスト定理）

- 人間の耳が聞こえる範囲：20 Hz 〜 **20,000 Hz**
- 必要なサンプリング周波数：20,000 × 2 = **40,000 Hz 以上**
- CD規格：**44,100 Hz**（余裕を持たせた値）

つまり `sample_rate=44100` は「人間に聞こえるすべての音を記録できる」設定です。

In [ ]:
# サンプリングの効果を体験してみよう
# ノコギリ波（高調波を多く含む音）で違いを聴き比べる

print("🔊 CD品質 (44100 Hz) — 豊かな倍音が聞こえる")
signal_high = sawtooth_wave(440, 1.0, sample_rate=44100) * 0.8
display(Audio(signal_high.data, rate=signal_high.sample_rate, normalize=False))

print("🔊 電話品質 (8000 Hz) — 高い倍音がカットされ、こもった音に")
signal_low = sawtooth_wave(440, 1.0, sample_rate=8000) * 0.8
display(Audio(signal_low.data, rate=signal_low.sample_rate, normalize=False))

print("\n💡 ノコギリ波は多くの倍音（440Hz, 880Hz, 1320Hz, ...）を含みます")
print("   電話品質(8000Hz)では 4000Hz 以上の倍音が記録できないため、")
print("   こもった音に聞こえます")
print(f"   CD品質: {len(signal_high)} サンプル")
print(f"   電話品質: {len(signal_low)} サンプル")
print("\n📝 ちなみに純粋なサイン波(440Hz)なら、どちらでも同じ音に聞こえます")
print("   （440Hz < 8000/2 = 4000Hz なので問題なく再現できる）")

In [ ]:
# ナイキスト定理の違反を体験
# 5000Hzの高い音を、低いサンプリング周波数で記録すると？

print("🔊 5000Hz を 44100Hz でサンプリング（正常）")
normal = sine_wave(5000, 1.0, sample_rate=44100)
display(Audio(normal.data, rate=normal.sample_rate))

print("🔊 5000Hz を 8000Hz でサンプリング（ナイキスト定理違反！）")
aliased = sine_wave(5000, 1.0, sample_rate=8000)
display(Audio(aliased.data, rate=aliased.sample_rate))

print("\n🚨 5000Hz > 8000/2 = 4000Hz なので正しく記録できません")
print("   元の音とは全く違う音に聞こえます（エイリアシング）")

## 🎯 実習2: 周波数を変えて音程を体験しよう

In [ ]:
# 異なる周波数の音を作ってみよう
frequencies = {
    "低い音 (220Hz)": 220,
    "基準音 (440Hz)": 440,
    "高い音 (880Hz)": 880,
    "とても高い音 (1760Hz)": 1760
}

duration = 1.5  # 少し長めに

for name, freq in frequencies.items():
    signal = sine_wave(freq, duration)
    audio_player = play_sound(signal, name)
    display(audio_player)
    
print("\n💡 気づいたこと:")
print("- 周波数が高いほど、音程が高くなる")
print("- 880Hzは440Hzの2倍 → 1オクターブ上の音")
print("- 1760Hzは440Hzの4倍 → 2オクターブ上の音")

## 🎯 実習3: 音量を変えてみよう

振幅（波の大きさ）を変えると音量が変わります。下のセルで実際に聴き比べてみましょう。

In [ ]:
# 同じ周波数で音量を変えてみよう
frequency = 440  # ラ音
duration = 1.0

volumes = {
    "とても小さい音 (0.1倍)": 0.1,
    "小さい音 (0.3倍)": 0.3,
    "大きい音 (0.8倍)": 0.8,
    "普通の音 (1.0倍)": 1.0,
}

for name, volume in volumes.items():
    signal = sine_wave(frequency, duration)
    signal_with_volume = signal * volume  # 音量調整
    
    audio_player = play_sound(signal_with_volume, f"{name} - 振幅: {volume}")
    display(audio_player)

print("\n💡 ポイント:")
print("・音量は波の振幅（高さ）で決まります")
print("・振幅0.1 → 0.3 → 0.8 → 1.0 と大きくなるにつれ、音量が上がります")

## 🎯 実習4: 音名と周波数の関係

In [10]:
# 音名から周波数を調べよう
note_names = ['C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5']
japanese_names = ['ド', 'レ', 'ミ', 'ファ', 'ソ', 'ラ', 'シ', 'ド']

print("🎹 音名と周波数の対応表:")
print("音名\t日本名\tMIDI番号\t周波数(Hz)")
print("-" * 40)

for note, japanese in zip(note_names, japanese_names):
    midi_number = note_name_to_number(note)
    frequency = note_to_frequency(midi_number)
    print(f"{note}\t{japanese}\t{midi_number}\t{frequency:.1f}")

🎹 音名と周波数の対応表:
音名	日本名	MIDI番号	周波数(Hz)
----------------------------------------
C4	ド	60	261.6
D4	レ	62	293.7
E4	ミ	64	329.6
F4	ファ	65	349.2
G4	ソ	67	392.0
A4	ラ	69	440.0
B4	シ	71	493.9
C5	ド	72	523.3


In [ ]:
# ドレミファソラシドを演奏しよう
print("🎵 ドレミファソラシドを演奏します！")

note_duration = 0.6  # 各音0.6秒

for i, (note, japanese) in enumerate(zip(note_names, japanese_names)):
    print(f"♪ {japanese} ({note})")
    
    # 音名をMIDI番号に変換
    midi_number = note_name_to_number(note)
    # MIDI番号を周波数に変換
    frequency = note_to_frequency(midi_number)
    
    # サイン波を生成
    signal = sine_wave(frequency, note_duration)
    audio = Audio(signal.data, rate=signal.sample_rate)
    display(audio)

print("\n🎉 これで基本的なスケール（音階）の完成です！")

## 🏆 チャレンジ課題

以下の課題に挑戦してみましょう！

In [ ]:
# チャレンジ1: 好きな周波数の音を作ってみよう
# ヒント: 200～2000Hzの範囲で試してみてください

my_frequency = 500  # ここを変更してください
my_duration = 2.0   # ここを変更してください

my_signal = sine_wave(my_frequency, my_duration)
my_audio = Audio(my_signal.data, rate=my_signal.sample_rate)

print(f"🎵 あなたの音: {my_frequency}Hz, {my_duration}秒")
display(my_audio)

In [ ]:
# チャレンジ2: 2つの音を同時に鳴らしてみよう
# これを「和音」と言います

freq1 = 440   # ラ音
freq2 = 554   # C#音（ラとハーモニーが美しい）
duration = 2.0

# 2つのサイン波を生成
signal1 = sine_wave(freq1, duration)
signal2 = sine_wave(freq2, duration)

# 2つの音を重ね合わせ
harmony = signal1 + signal2
# 音量を調整（2つ重ねると大きくなるため）
harmony = harmony * 0.5

print(f"🎵 和音: {freq1}Hz + {freq2}Hz")
harmony_audio = Audio(harmony.data, rate=harmony.sample_rate)
display(harmony_audio)

print("💡 2つの音が同時に聞こえて、豊かな響きになりましたね！")

## 📚 今日のまとめ

### 学んだこと
1. **音の正体**: 空気の振動、波として表現
2. **サイン波**: 最も基本的な音の波形
3. **サンプリング**: 音をデジタルデータにする仕組み
4. **ナイキスト定理**: サンプリング周波数は最高周波数の2倍以上必要
5. **周波数**: 音程を決める重要な要素
6. **振幅**: 音量を決める要素
7. **音名**: C4、D4などの音楽的な表現
8. **和音**: 複数の音を同時に鳴らす

### 使ったライブラリ
- `sine_wave()`: サイン波の生成
- `AudioSignal`: 音声データを扱うオブジェクト
- `save_audio()`: WAVファイルの保存
- `note_to_frequency`: 音名→周波数の変換

### 次回予告
次回は「**エンベロープ**」を学びます。
音が時間とともにどう変化するかをコントロールして、より自然で楽器らしい音を作ります！

---
**お疲れさまでした！** 🎉